<a href="https://colab.research.google.com/github/vitroid/PythonTutorials/blob/master/8OtherClasses/MolecularOrbital.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Google Colaboratory上で分子軌道計算を試します。

[Labo-code](https://labo-code.com/python/quantum-chemical-calculation/moleculaorbital/)の記事を参考にしました。感謝!


# 1. 必要なパッケージのインストール


In [ ]:
!pip install pyscf
!pip install numpy
!pip install geometric
!pip install py3Dmol

# 2. 初期データを作る

Labo-codeさんの記事をそのまま使っても構いませんが、ここではベンゼンではなく水分子を作ってみます。

水分子の正確な原子位置はわからない(むしろ計算した結果として得たい)ので、ベンゼンの炭素3つの座標を酸素と水素に使います。



In [ ]:
from pyscf import gto

# 水分子の定義 (座標の単位はÅ、すごくてきとう。)
mol = gto.M(
    atom='''
H       -0.7 0.9 0
O       0 0 0
H       0.7 0.9 0
    ''',
    basis='6-31G(d)',
    spin=0,  # スピンを指定
    charge=0,  # 電荷を指定
    verbose=0
)


In [ ]:
# とりあえず、そのまま表示してみる
from pyscf import tools
import py3Dmol
from IPython.display import display, HTML
import ipywidgets as widgets

def mol_to_xyz(mol):
    geometry = mol.atom_coords(unit="ANG")
    xyz = "\n".join([f"{mol.atom_symbol(i)} {geometry[i][0]:.6f} {geometry[i][1]:.6f} {geometry[i][2]:.6f}" for i in range(mol.natm)])
    xyz = f"{mol.natm}\n\n" + xyz +"\n" # xyzフォーマットのヘッダー追加
    return xyz

def view_xyz(view, xyz):
    view.addModel(xyz, 'xyz')
    view.setStyle({
        'stick': {'radius': 0.2},
        'sphere': {'scale': 0.3,}
    })
    view.zoomTo()
    return view

xyz = mol_to_xyz(mol)
view = py3Dmol.view(width=800, height=600)
view = view_xyz(view, xyz)
display(HTML(view._make_html()))


# 3. 構造最適化

与えられた原子核の初期配置に対し、電子を配置してエネルギーができるだけ低くなるように電子と原子核の配置を最適にします。これを構造最適化と呼びます。


In [ ]:
from pyscf.geomopt import geometric_solver
from pyscf import dft, scf

# # DFT計算のセットアップ
# mf = dft.RKS(mol)

# SCF
mf = scf.RHF(mol)

mf.chkfile = 'checkpoint_file.chk'
# mf.xc = 'B3LYP'
mf.max_cycle = 150  # SCF計算の最大反復回数

# 構造最適化
mol_eq = geometric_solver.optimize(mf, maxsteps=100)


In [ ]:
xyz = mol_to_xyz(mol_eq)
view = py3Dmol.view(width=800, height=600)
view = view_xyz(view, xyz)
display(HTML(view._make_html()))


# 4. 電子配置

電子を、エネルギーの低い軌道から順に入れていきます。酸素と水素で全部で10個の電子があるので、5つの軌道が埋まります。

エネルギーの単位をhartreeからeVに換算して表示します。

In [ ]:
# # 最適化後のジオメトリーを取得
# final_geometry = mol.atom_coords(unit="ANG")

# # SCF計算の再実行
# mf = dft.RKS(mol)
# mf.chkfile = 'checkpoint_file.chk'
# mf.xc = 'B3LYP'
mf.kernel()

# エネルギーをeVに変換
hartree_to_ev = 27.2114

mf.mo_occ, mf.mo_energy * hartree_to_ev


電子が入っている、もっともエネルギーの高い(最も外側にある)軌道のことをHOMO (Highest Occupied Molecular Orbital; 最高被占分子軌道、その一つ上の軌道のことをLUMO (Lowest Unoccupied Molecular Orbital; 最低空分子軌道)と呼びます。一番外側の軌道は、分子の形や大きさや反応性を決定づけるので、化学ではこれらの軌道に特に注目します。

水の場合、HOMOは5番目、LUMOは6番目の軌道です。

Pythonでは、数字を0から数えはじめるので、これらはそれぞれ4番目、5番目に相当します。

In [ ]:
homo_index, lumo_index = 4, 5

# 5. 軌道の可視化

この部分は、Labo-codeからそのままもってきました。

In [ ]:
from pyscf import tools
import py3Dmol
from IPython.display import display, HTML
import ipywidgets as widgets

# 指定した軌道のCUBEファイルを生成する関数
def generate_cube_file(orbital_index, filename):
    tools.cubegen.orbital(mol, filename, mf.mo_coeff[:, orbital_index])

# 可視化する軌道のリストを作成
orbitals = {
    "LUMO+2": lumo_index + 2,
    "LUMO+1": lumo_index + 1,
    "LUMO": lumo_index,
    "HOMO": homo_index,
    "HOMO-1": homo_index - 1,
    "HOMO-2": homo_index - 2,
    "HOMO-3": homo_index - 3,
    "HOMO-4": homo_index - 4
}

# エネルギーをeVに変換
hartree_to_ev = 27.2114
orbital_energies = {name: mf.mo_energy[idx] * hartree_to_ev for name, idx in orbitals.items()}

# ウィジェットの作成
dropdown = widgets.Dropdown(
    options=[(f"{name} ({orbital_energies[name]:.6f} eV)", name) for name in orbitals.keys()],
    description='Orbital:',
    disabled=False,
)

output = widgets.Output()

# 可視化関数
def visualize_orbital(orbital_name):
    orbital_index = orbitals[orbital_name]
    cube_filename = f'{orbital_name}.cube'

    generate_cube_file(orbital_index, cube_filename)

    view = py3Dmol.view(width=800, height=600)
    with open(cube_filename) as f:
        cube = f.read()
    view.addVolumetricData(cube, 'cube', {'isoval': 0.02, 'color': 'red', 'opacity': 0.75})
    view.addVolumetricData(cube, 'cube', {'isoval': -0.02, 'color': 'blue', 'opacity': 0.75})

    # 最適化後の分子形状を追加
    xyz = mol_to_xyz(mol_eq)
    view = view_xyz(view, xyz)

    # エネルギーのラベルを追加
    orbital_energy = orbital_energies[orbital_name]
    view.addLabel(f'{orbital_name}: {orbital_energy:.6f} eV', {'fontSize': 12, 'fontColor': 'black', 'backgroundColor': 'white', 'backgroundOpacity': 0.8, 'showBackground': True, 'position': {'x': 0, 'y': 0, 'z': 0}})

    with output:
        output.clear_output(wait=True)
        display(HTML(view._make_html()))

# ドロップダウンの変更イベントに可視化関数をバインド
def on_dropdown_change(change):
    with output:
        output.clear_output(wait=True)
        visualize_orbital(change['new'])

dropdown.observe(on_dropdown_change, names='value')

# インターフェースの表示
display(dropdown)
display(output)

# 初期表示
visualize_orbital(list(orbitals.keys())[0])
